# 01 - 特征工程技巧

## 学习目标
- 掌握常用特征工程方法
- 学会创建和选择特征
- 理解特征缩放的重要性

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    StandardScaler, PolynomialFeatures,
    QuantileTransformer, PowerTransformer
)
from sklearn.feature_selection import (
    SelectKBest, f_regression, mutual_info_regression,
    RFE, SelectFromModel
)
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

## 1. 多项式特征

In [ ]:
# 创建简单数据
np.random.seed(42)
X = np.random.rand(100, 2)
y = 3 * X[:, 0]**2 + 2 * X[:, 0] * X[:, 1] + X[:, 1] + np.random.randn(100) * 0.1

# 多项式特征
poly = PolynomialFeatures(degree=2, interaction_only=False, include_bias=False)
X_poly = poly.fit_transform(X)

print(f'原始特征: {X.shape[1]}')
print(f'多项式特征: {X_poly.shape[1]}')
print(f'特征名称: {poly.get_feature_names_out(["x1", "x2"])}')

## 2. 特征选择

In [ ]:
# 加载加州房价数据
data = fetch_california_housing()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# 基于统计检验的特征选择
selector = SelectKBest(score_func=f_regression, k=5)
X_selected = selector.fit_transform(X, y)

scores = pd.DataFrame({
    'feature': X.columns,
    'score': selector.scores_
}).sort_values('score', ascending=False)

print('特征得分排名:')
print(scores.to_string(index=False))

In [ ]:
# 基于模型的特征选择 (RFE)
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rfe = RFE(estimator=rf, n_features_to_select=5)
rfe.fit(X, y)

selected_features = X.columns[rfe.support_]
print(f'RFE 选择的特征: {list(selected_features)}')

# 特征重要性排名
ranking = pd.DataFrame({
    'feature': X.columns,
    'ranking': rfe.ranking_
}).sort_values('ranking')
print('\n特征排名:')
print(ranking.to_string(index=False))

## 3. 特征变换

In [ ]:
# 对比不同缩放方法
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scalers = {
    'StandardScaler': StandardScaler(),
    'QuantileTransformer': QuantileTransformer(output_distribution='normal'),
    'PowerTransformer': PowerTransformer(method='yeo-johnson'),
}

results = []
for name, scaler in scalers.items():
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    model = LinearRegression()
    model.fit(X_train_scaled, y_train)
    score = model.score(X_test_scaled, y_test)
    results.append({'Scaler': name, 'R²': score})

print(pd.DataFrame(results).round(4))

## 小结
- 多项式特征可以捕捉非线性关系
- 特征选择减少维度，提高模型性能
- 不同的缩放方法适用于不同场景